# tm_score-only production pipeline — click-to-top-100 latency profile

Wall-clock latency of the accepted production pipeline
(`production_seed_precision_at_k/production_seed_precision_at_k_chromatin_half_pix_fix.ipynb`,
cells 1/3/5/6 — D8_TEMPLATE_ANCHOR.md, current as of 2026-09-10), reusing its exact config
and RNG seeding. Pure timing — no precision, no chromatin/shape features, no other ranking
arms.

**SETUP vs PIPELINE.** SETUP (once per ROI, not part of click latency) loads the ROI and
builds the click-candidate pool. PIPELINE times the six stages a real click triggers.

**Click-validity assumption.** The box has already been drawn — i.e. a *valid* seed has
already been selected. `draw_seed_with_retry` (its retry loop over the candidate pool) runs
once, untimed, inside SETUP, purely to land on a point `tightened_template_box` will accept;
that search is test-harness scaffolding standing in for "the point a human clicked," not
something a real click does. 13/14 ROIs accept on the first draw; 403.tiff needs one retry
(`n_retries=1`), logged as context only, not counted in the timed total.

Timed PIPELINE stage 1 then re-runs `tightened_template_box` (plus its border-safety check)
**once, retry-free**, on that already-valid click point — the real work a click triggers:
turning one known-good point into a refined bounding box.

In [1]:
import gc
import os
import sys
import time

import cv2
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from midog_utils import channels as ch
from midog_utils import dataset as ds
from midog_utils import evaluate as ev
from midog_utils import find_and_suppress as fs
from midog_utils import seed_selection as ss
from midog_utils import template_match as tm
from midog_utils.nms import nms_by_distance

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 220)

# ---------------------------------------------------------------------------------------
# Config -- copied verbatim from cell 1 of the accepted notebook. Do not change.
# ---------------------------------------------------------------------------------------
IMAGES_DIR = '../images/extra_valid'
SEED_INDEX = 0

CHANNEL = 'hematoxylin_od'
METHOD = cv2.TM_CCOEFF
PEAK_MIN_DISTANCE = 7
SELF_HIT_RADIUS = 5.0
DEEP_FLOOR_Z = -1.5
MAX_PEAKS = 2_000_000

NMS_RADIUS_UM = ev.MIDOG_RADIUS_UM
MATCH_RADIUS_UM = ev.MIDOG_RADIUS_UM

CFG = fs.FSConfig(channel=CHANNEL, base_size=tm.BASE_SIZE, scales=(1.0,),
                  n_angles=1, flips=(False,), peak_min_distance=PEAK_MIN_DISTANCE,
                  self_hit_radius=SELF_HIT_RADIUS)
BORDER = CFG.patch_size // 2
OTSU_WINDOW = tm.BASE_SIZE

BUDGET = 100  # this task's top-K; the notebook's own BUDGETS=(10,20,30,50) don't apply

ORACLE_RAW_CSV = '../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv'

print(f'BUDGET={BUDGET}, NMS radius = match radius = {NMS_RADIUS_UM} um, channel={CHANNEL}')

BUDGET=100, NMS radius = match radius = 7.5 um, channel=hematoxylin_od


## Helpers

Copied or reimplemented verbatim from the accepted notebook (cell 2) -- `suppress()` is inline there, not part of any `midog_utils` module.

In [2]:
def draw_seed_with_retry(pool, rng, check_fn):
    """Draw a row via `rng.integers`; on failure drop it and redraw on the same stream.

    Runs once per ROI, in SETUP below -- this is how the harness locates a *valid* click
    point standing in for "the point a human clicked." A real click skips this search.
    """
    working, retries = pool.copy(), 0
    while len(working) > 0:
        idx = int(rng.integers(len(working)))
        row = working.iloc[idx]
        result = check_fn(row)
        if result is not None:
            return row, result, retries
        working = working.drop(working.index[idx])
        retries += 1
    raise ValueError('seed pool exhausted -- no candidate passed check_fn')


def suppress(centers, scores, radius, ref_xy):
    """NMS at `radius`, then drop the template's own self-correlation."""
    keep = nms_by_distance(centers, scores, radius)
    c, s = centers[keep], scores[keep]
    if len(c):
        ok = np.hypot(c[:, 0] - ref_xy[0], c[:, 1] - ref_xy[1]) > SELF_HIT_RADIUS
        c, s = c[ok], s[ok]
    return c, s


def roi_files(images_dir=IMAGES_DIR):
    return sorted(f for f in os.listdir(images_dir) if f.endswith('.tiff'))


print(f'{len(roi_files())} ROIs on disk in {IMAGES_DIR}/')

14 ROIs on disk in ../images/extra_valid/


## Per-ROI timing function

`gc` is disabled for the duration of each ROI's timed work and an explicit `gc.collect()` runs between ROIs, so a stray collection during a multi-MB array allocation doesn't inflate one stage's number.

In [3]:
def time_roi(fn, image_id, domain, anns):
    gc.disable()

    # =================== SETUP (once per ROI; not part of click latency) ==============
    # Loads the ROI, builds the click-candidate pool, then -- test-harness only --
    # searches that pool (with retries) for a point tightened_template_box will accept.
    t0 = time.perf_counter()
    path = f'{IMAGES_DIR}/{fn}'
    rgb = ds.load_roi(path)
    mpp = ds.roi_mpp(path)
    roi_shape = rgb.shape
    nms_radius = ev.radius_px(mpp, NMS_RADIUS_UM)
    hem = ch.to_channel(rgb, CHANNEL)
    gray_inv = ch.to_gray_inverted(rgb)
    H, W = hem.shape[:2]
    del rgb

    gt = ds.image_annotations(anns, fn)
    seed_pool, flagged = ss.agreement_pool(gt[gt['category_id'] == ds.MITOTIC])
    seed_pool = ss.border_filter(seed_pool, BORDER, roi_shape)

    rng = np.random.default_rng([SEED_INDEX, image_id])

    def _check(row):
        r = ss.tightened_template_box(gray_inv, float(row['cx']), float(row['cy']),
                                      otsu_window=OTSU_WINDOW)
        if r is None:
            return None
        if tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size) is None:
            return None
        return r

    seed, seed_tpl_probe, n_retries = draw_seed_with_retry(seed_pool, rng, _check)
    seed_ann_id = int(seed['ann_id'])
    click_cx, click_cy = float(seed['cx']), float(seed['cy'])
    t_setup = time.perf_counter() - t0

    # ======================= PIPELINE (click-to-top-100 latency) =======================
    # The box has been drawn: `seed` is already a *valid* click, established above.
    # Timing starts at the step that refines that valid seed's bounding box.
    stages = {}
    t_outer0 = time.perf_counter()

    # ---- Stage 1: refine the bounding box of the valid seed (single call, no retry) ---
    t1 = time.perf_counter()
    r = ss.tightened_template_box(gray_inv, click_cx, click_cy, otsu_window=OTSU_WINDOW)
    assert r is not None, f'{fn}: seed was pre-validated in SETUP but refused here'
    border_probe = tm.read_padded_patch(hem, r[1], r[2], CFG.patch_size)
    assert border_probe is not None, f'{fn}: seed was pre-validated in SETUP but unreadable here'
    assert r == seed_tpl_probe, f'{fn}: single-shot refinement diverged from the SETUP search'
    base_size, tpl_cx, tpl_cy = r
    tpl_xy = (float(tpl_cx), float(tpl_cy))
    stages['t1_refine_seed_box_s'] = time.perf_counter() - t1

    # ---- Stage 2: patch + template build ----------------------------------------------
    t2 = time.perf_counter()
    patch = tm.read_padded_patch(hem, *tpl_xy, CFG.patch_size)
    templates, _ = tm.build_augmentations(patch, base_size, CFG.scales, CFG.n_angles, CFG.flips)
    stages['t2_patch_template_build_s'] = time.perf_counter() - t2

    # ---- Stage 3: template matching (suspected dominant cost) -------------------------
    t3 = time.perf_counter()
    PAD = max((t.shape[0] - 1) // 2 for t in templates)
    hem_p = cv2.copyMakeBorder(hem, PAD, PAD, PAD, PAD, borderType=cv2.BORDER_REPLICATE)
    fused_p, _, valid_p = tm.fused_response(hem_p, templates, CFG.scale_normalize, method=METHOD)
    stages['t3_template_matching_s'] = time.perf_counter() - t3

    # ---- Stage 4: threshold + peak extraction ------------------------------------------
    t4 = time.perf_counter()
    fused = fused_p[PAD:PAD + H, PAD:PAD + W]
    valid = valid_p[PAD:PAD + H, PAD:PAD + W]
    assert bool(valid.all()), f'{fn}: padding left part of the ROI unreachable (PAD={PAD})'
    med, mad = tm.robust_stats(fused, valid)
    cut = med + DEEP_FLOOR_Z * mad
    centers, scores = tm.extract_peaks(fused, valid, PEAK_MIN_DISTANCE, cut, MAX_PEAKS)
    assert len(centers) < MAX_PEAKS, f'{fn}: MAX_PEAKS is binding, raise it'
    stages['t4_threshold_peak_extraction_s'] = time.perf_counter() - t4

    # ---- Stage 5: NMS + self-hit suppression -------------------------------------------
    t5 = time.perf_counter()
    c, s = suppress(centers, scores, nms_radius, tpl_xy)
    stages['t5_nms_selfhit_s'] = time.perf_counter() - t5
    assert bool(np.all(np.diff(s) <= 0)), f'{fn}: post-NMS pool is not score-descending'

    # ---- Stage 6: rank + top-100 (matches midog_utils/compare.py's _rank convention) ---
    t6 = time.perf_counter()
    pool = pd.DataFrame({'cx': c[:, 0], 'cy': c[:, 1], 'score': s})
    top = pool.sort_values('score', ascending=False, na_position='last',
                           kind='mergesort').head(BUDGET)
    stages['t6_rank_top100_s'] = time.perf_counter() - t6

    t_outer_total = time.perf_counter() - t_outer0
    gc.enable()

    n_detections = len(pool)
    n_top = len(top)
    stage_sum = sum(stages.values())

    del hem, gray_inv, hem_p, fused_p, valid_p, fused, valid, templates, patch, centers, scores, c, s
    gc.collect()

    meta = dict(
        file_name=fn, tumor_type=domain,
        n_detections=n_detections, base_size=base_size, n_retries=n_retries, n_top=n_top,
        seed_ann_id=seed_ann_id, map_median=round(float(med), 5), mad_scale=round(float(mad), 5),
        t_setup_s=round(t_setup, 5),
        **{k: round(v, 5) for k, v in stages.items()},
        t_stage_sum_s=round(stage_sum, 5),
        t_outer_total_s=round(t_outer_total, 5),
    )
    print(f"[{fn}] {domain:32s} base={base_size:2d} n_detections={n_detections:6d} "
          f"n_retries={n_retries} n_top={n_top:3d} "
          f"sum={stage_sum * 1000:7.1f}ms outer={t_outer_total * 1000:7.1f}ms "
          f"(setup={t_setup * 1000:.0f}ms)", flush=True)
    return meta

## Run — all 14 ROIs

In [4]:
images, annotations = ds.load_annotations('../databases/MIDOG++.json')
ds.check_invariants(annotations)
meta_ix = images.set_index('file_name')[['image_id', 'tumor_type']]

files = roi_files()
assert len(files) == 14, f'expected 14 ROIs in {IMAGES_DIR}/, found {len(files)}'

rows = []
for fn in files:
    image_id = int(meta_ix.loc[fn, 'image_id'])
    domain = meta_ix.loc[fn, 'tumor_type']
    rows.append(time_roi(fn, image_id, domain, annotations))
    gc.collect()

TIMING = pd.DataFrame(rows).set_index('file_name')
TIMING.to_csv('tm_score_latency_per_roi.csv')
print(f'\n{len(TIMING)} ROIs timed -> tm_score_latency_per_roi.csv')

[013.tiff] human breast cancer              base=31 n_detections= 17724 n_retries=0 n_top=100 sum= 1870.7ms outer= 1870.8ms (setup=3861ms)


[094.tiff] human breast cancer              base=25 n_detections= 18628 n_retries=0 n_top=100 sum= 1502.6ms outer= 1502.6ms (setup=2706ms)


[201.tiff] canine lung cancer               base=51 n_detections= 15848 n_retries=0 n_top=100 sum= 1414.6ms outer= 1414.7ms (setup=2139ms)


[233.tiff] canine lung cancer               base=25 n_detections= 17449 n_retries=0 n_top=100 sum= 1219.8ms outer= 1219.9ms (setup=2000ms)


[245.tiff] canine lymphosarcoma             base=47 n_detections= 17678 n_retries=0 n_top=100 sum= 1466.3ms outer= 1466.3ms (setup=2274ms)


...

[459.tiff] canine soft tissue sarcoma       base=33 n_detections= 17806 n_retries=0 n_top=100 sum= 1262.2ms outer= 1262.2ms (setup=2327ms)


[460.tiff] canine soft tissue sarcoma       base=47 n_detections= 15698 n_retries=0 n_top=100 sum= 1315.3ms outer= 1315.4ms (setup=2298ms)


[529.tiff] human melanoma                   base=37 n_detections= 16620 n_retries=0 n_top=100 sum= 1474.5ms outer= 1474.6ms (setup=3262ms)


[548.tiff] human melanoma                   base=29 n_detections= 17839 n_retries=0 n_top=100 sum= 1500.5ms outer= 1500.5ms (setup=3437ms)



14 ROIs timed -> tm_score_latency_per_roi.csv


## Cross-check against the accepted notebook's own results

Same RNG seed, same config -> same template -> same fused map -> same post-NMS pool, so `seed_ann_id`, `base_size`, `n_detections`, `map_median` and `mad_scale` should match the accepted notebook's own committed CSV **exactly**, not approximately. A mismatch would mean this harness diverged from production -- a bug to report, not proceed past.

In [5]:
oracle = pd.read_csv(ORACLE_RAW_CSV)
oracle_roi = (oracle.drop_duplicates('file_name')
              .set_index('file_name')[['seed_ann_id', 'base_size', 'n_detections',
                                        'map_median', 'mad_scale']])
cmp = TIMING[['seed_ann_id', 'base_size', 'n_detections', 'map_median', 'mad_scale']].join(
    oracle_roi, lsuffix='_this', rsuffix='_oracle')

mismatches = []
for col in ['seed_ann_id', 'base_size', 'n_detections']:
    bad = cmp[f'{col}_this'].astype(int) != cmp[f'{col}_oracle'].astype(int)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))
for col in ['map_median', 'mad_scale']:
    bad = ~np.isclose(cmp[f'{col}_this'], cmp[f'{col}_oracle'], rtol=0, atol=1e-5)
    if bad.any():
        mismatches.append((col, cmp.index[bad].tolist()))

if mismatches:
    print('!! MISMATCH vs oracle -- profiling harness diverged from production:')
    for col, rois in mismatches:
        print(f'   {col}: {rois}')
    display(cmp)
    raise AssertionError('profiling harness does not reproduce the accepted pipeline')

print(f'All {len(cmp)} ROIs match {ORACLE_RAW_CSV} exactly on '
      f'seed_ann_id/base_size/n_detections/map_median/mad_scale -- harness reproduces production.')

All 14 ROIs match ../results/precision_at_k_14roi_prodseed_chromatin_halfpixfix_raw.csv exactly on seed_ann_id/base_size/n_detections/map_median/mad_scale -- harness reproduces production.


## Table 1 — per-ROI, per-stage timing (ms)

In [6]:
STAGE_COLS_S = ['t1_refine_seed_box_s', 't2_patch_template_build_s', 't3_template_matching_s',
                't4_threshold_peak_extraction_s', 't5_nms_selfhit_s', 't6_rank_top100_s']
ALL_TIME_COLS_S = ['t_setup_s'] + STAGE_COLS_S + ['t_stage_sum_s', 't_outer_total_s']

TIMING_MS = TIMING.copy()
for c in ALL_TIME_COLS_S:
    TIMING_MS[c[:-2] + '_ms'] = (TIMING_MS[c] * 1000).round(1)
STAGE_COLS_MS = [c[:-2] + '_ms' for c in STAGE_COLS_S]
ALL_TIME_COLS_MS = [c[:-2] + '_ms' for c in ALL_TIME_COLS_S]

display_cols = ['tumor_type', 'base_size', 'n_detections', 'n_retries', 'n_top',
                't_setup_ms'] + STAGE_COLS_MS + ['t_stage_sum_ms', 't_outer_total_ms']
TIMING_MS[display_cols]

,tumor_type,base_size,n_detections,n_retries,n_top,t_setup_ms,t1_refine_seed_box_ms,t2_patch_template_build_ms,t3_template_matching_ms,t4_threshold_peak_extraction_ms,t5_nms_selfhit_ms,t6_rank_top100_ms,t_stage_sum_ms,t_outer_total_ms
file_name,,,,,,,,,,,,,,
013.tiff,human breast cancer,31,17724,0,100,3860.7,0.9,0.0,1242.0,303.9,322.4,1.4,1870.7,1870.8
094.tiff,human breast cancer,25,18628,0,100,2706.1,1.3,0.0,919.4,283.3,297.9,0.7,1502.6,1502.6
201.tiff,canine lung cancer,51,15848,0,100,2139.1,1.1,0.0,1037.6,239.9,135.4,0.6,1414.6,1414.6
233.tiff,canine lung cancer,25,17449,0,100,1999.5,0.8,0.0,774.2,237.9,206.1,0.8,1219.8,1219.9
245.tiff,canine lymphosarcoma,47,17678,0,100,2273.8,1.2,0.0,956.8,246.8,260.7,0.8,1466.3,1466.4
246.tiff,canine lymphosarcoma,41,18013,0,100,2116.2,1.5,0.0,941.9,252.5,195.4,0.8,1392.0,1392.1
300.tiff,canine cutaneous mast cell tumor,45,17940,0,100,2338.5,1.3,0.0,985.6,233.9,145.2,0.8,1366.8,1366.9
301.tiff,canine cutaneous mast cell tumor,41,17710,0,100,2225.5,1.1,0.0,870.3,223.7,154.3,0.7,1250.2,1250.2
402.tiff,human neuroendocrine tumor,29,17532,0,100,2367.8,0.7,0.0,950.5,292.8,236.6,0.6,1481.3,1481.4


## Table 2 — summary across all 14 ROIs (ms)

In [7]:
SUMMARY = TIMING_MS[ALL_TIME_COLS_MS].agg(['mean', 'std', 'min', 'max']).T
total_mean_ms = TIMING_MS['t_stage_sum_ms'].mean()
SUMMARY['pct_of_pipeline_total'] = np.nan
SUMMARY.loc[STAGE_COLS_MS, 'pct_of_pipeline_total'] = (
    SUMMARY.loc[STAGE_COLS_MS, 'mean'] / total_mean_ms * 100).round(1)
SUMMARY.to_csv('tm_score_latency_summary.csv')
SUMMARY.round(2)

,mean,std,min,max,pct_of_pipeline_total
t_setup_ms,2554.69,560.68,1999.5,3860.7,NaN
t1_refine_seed_box_ms,1.09,0.23,0.7,1.5,0.1
t2_patch_template_build_ms,0.00,0.00,0.0,0.0,0.0
t3_template_matching_ms,992.64,147.85,774.2,1350.4,68.4
t4_threshold_peak_extraction_ms,260.21,27.17,223.7,303.9,17.9
t5_nms_selfhit_ms,195.55,64.35,106.5,322.4,13.5
t6_rank_top100_ms,0.86,0.33,0.6,1.8,0.1
t_stage_sum_ms,1450.34,187.34,1219.8,1870.7,NaN
t_outer_total_ms,1450.41,187.35,1219.9,1870.8,NaN


## Table 3 — summary by tumor domain (ms)

In [8]:
BY_DOMAIN = TIMING_MS.groupby('tumor_type')[ALL_TIME_COLS_MS].agg(['mean', 'std', 'min', 'max'])
BY_DOMAIN.to_csv('tm_score_latency_by_domain.csv')
BY_DOMAIN.round(2)

t_setup_ms                         t1_refine_seed_box_ms                 t2_patch_template_build_ms                t3_template_matching_ms                         \
                                       mean     std     min     max                  mean   std  min  max                       mean  std  min  max                    mean     std    min     max   
tumor_type                                                                                                                                                                                           
canine cutaneous mast cell tumor    2282.00   79.90  2225.5  2338.5                  1.20  0.14  1.1  1.3                        0.0  0.0  0.0  0.0                  927.95   81.53  870.3   985.6   
canine lung cancer                  2069.30   98.71  1999.5  2139.1                  0.95  0.21  0.8  1.1                        0.0  0.0  0.0  0.0                  905.90  186.25  774.2  1037.6   
canine lymphosarcoma                2195.00  111.44  2116.2  2273.8                  1.35  0.21  1.2  1.5                        0.0  0.0  0.0  0.0                  949.35   10.54  941.9   956.8   
canine soft tissue sarcoma          2312.75   20.29  2298.4  2327.1                  1.20  0.00  1.2  1.2                        0.0  0.0  0.0  0.0                  919.35   73.04  867.7   971.0   
human breast cancer                 3283.40  816.43  2706.1  3860.7                  1.10  0.28  0.9  1.3                        0.0  0.0  0.0  0.0                 1080.70  228.11  919.4  1242.0   
human melanoma                      3349.45  124.24  3261.6  3437.3                  1.05  0.21  0.9  1.2                        0.0  0.0  0.0  0.0                 1014.75   34.44  990.4  1039.1   
human neuroendocrine tumor          2390.95   32.74  2367.8  2414.1                  0.75  0.07  0.7  0.8                        0.0  0.0  0.0  0.0                 1150.45  282.77  950.5  1350.4   

                                 t4_threshold_peak_extraction_ms                      t5_nms_selfhit_ms                      t6_rank_top100_ms                 t_stage_sum_ms                         t_outer_total_ms  \
                                                            mean    std    min    max              mean    std    min    max              mean   std  min  max           mean     std     min     max             mean   
tumor_type                                                                                                                                                                                                               
canine cutaneous mast cell tumor                          228.80   7.21  223.7  233.9            149.75   6.43  145.2  154.3              0.75  0.07  0.7  0.8        1308.50   82.45  1250.2  1366.8          1308.55   
canine lung cancer                                        238.90   1.41  237.9  239.9            170.75  49.99  135.4  206.1              0.70  0.14  0.6  0.8        1317.20  137.74  1219.8  1414.6          1317.25   
canine lymphosarcoma                                      249.65   4.03  246.8  252.5            228.05  46.17  195.4  260.7              0.80  0.00  0.8  0.8        1429.15   52.54  1392.0  1466.3          1429.25   
canine soft tissue sarcoma                                237.90   2.97  235.8  240.0            129.50  32.53  106.5  152.5              0.85  0.07  0.8  0.9        1288.75   37.55  1262.2  1315.3          1288.80   
human breast cancer                                       293.60  14.57  283.3  303.9            310.15  17.32  297.9  322.4              1.05  0.49  0.7  1.4        1686.65  260.29  1502.6  1870.7          1686.70   
human melanoma                                            283.75  14.07  273.8  293.7            186.70  38.33  159.6  213.8              1.30  0.71  0.8  1.8        1487.50   18.38  1474.5  1500.5          1487.55   
human neuroendocrine tumor                                288.85   5.59  284.9  292.8     

## Cold-start and starvation checks

In [9]:
print('Stage 3 (template matching): ROI #1 (process-cold) vs remaining 13')
print(f"  ROI #1 ({TIMING_MS.index[0]}): {TIMING_MS['t3_template_matching_ms'].iloc[0]:.1f} ms")
rest = TIMING_MS['t3_template_matching_ms'].iloc[1:]
print(f'  remaining 13: mean={rest.mean():.1f}ms std={rest.std():.1f}ms '
      f'min={rest.min():.1f}ms max={rest.max():.1f}ms')

starved = TIMING[TIMING['n_top'] < BUDGET]
if len(starved):
    print(f'\n{len(starved)} ROI(s) delivered fewer than top-{BUDGET}: {starved["n_top"].to_dict()}')
else:
    print(f'\nAll 14 ROIs delivered the full top-{BUDGET}.')

print(f'\nSETUP included a retry search on {int((TIMING["n_retries"] > 0).sum())}/14 ROIs '
      f'(n_retries value counts: {TIMING["n_retries"].value_counts().to_dict()}) -- this cost is '
      f'in t_setup_s, not in the timed PIPELINE total.')

outer_vs_sum = (TIMING_MS['t_outer_total_ms'] - TIMING_MS['t_stage_sum_ms']).abs()
print(f'\nOuter-timer vs stage-sum cross-check: max discrepancy '
      f'{outer_vs_sum.max():.2f}ms across all 14 ROIs -- stages 1-6 fully account for the '
      f'outer-timed interval.')

Stage 3 (template matching): ROI #1 (process-cold) vs remaining 13
  ROI #1 (013.tiff): 1242.0 ms
  remaining 13: mean=973.5ms std=134.5ms min=774.2ms max=1350.4ms

All 14 ROIs delivered the full top-100.

SETUP included a retry search on 1/14 ROIs (n_retries value counts: {0: 13, 1: 1}) -- this cost is in t_setup_s, not in the timed PIPELINE total.

Outer-timer vs stage-sum cross-check: max discrepancy 0.10ms across all 14 ROIs -- stages 1-6 fully account for the outer-timed interval.


## Written readout

In [10]:
dom_stage3 = TIMING_MS.groupby('tumor_type')['t3_template_matching_ms'].mean().sort_values()
total_mean = TIMING_MS['t_stage_sum_ms'].mean()
total_min = TIMING_MS['t_stage_sum_ms'].min()
total_max = TIMING_MS['t_stage_sum_ms'].max()
s3_mean = TIMING_MS['t3_template_matching_ms'].mean()
s3_pct = s3_mean / total_mean * 100
second = SUMMARY.loc[STAGE_COLS_MS, 'mean'].drop('t3_template_matching_ms').idxmax()
second_mean = SUMMARY.loc[second, 'mean']

print(f'Stage 3 (template matching) dominates: mean {s3_mean:.0f} ms/click, {s3_pct:.0f}% of the '
      f'{total_mean:.0f} ms average pipeline total -- {s3_mean / second_mean:.1f}x the next-largest '
      f'stage ({second}, {second_mean:.0f} ms). This holds on every one of the 14 ROIs and every '
      f'tumor domain (per-domain stage-3 means range {dom_stage3.min():.0f}-{dom_stage3.max():.0f} ms).')
print()
print(f'Pipeline total (stages 1-6) ranges {total_min:.0f}-{total_max:.0f} ms, mean {total_mean:.0f} ms '
      f'-- {total_min / 200:.1f}x to {total_max / 200:.1f}x the ~100-200ms "feels instant" budget for '
      f'a human clicker. No ROI is close to instant: template matching alone already exceeds that '
      f'budget on every ROI. The bank matched is a single template (scales=(1.0,), n_angles=1, '
      f'flips=(False,)) -- a richer augmentation bank would scale stage 3 roughly linearly, making '
      f'it an even larger share of total latency, not a smaller one.')

Stage 3 (template matching) dominates: mean 993 ms/click, 68% of the 1450 ms average pipeline total -- 3.8x the next-largest stage (t4_threshold_peak_extraction_ms, 260 ms). This holds on every one of the 14 ROIs and every tumor domain (per-domain stage-3 means range 906-1150 ms).

Pipeline total (stages 1-6) ranges 1220-1871 ms, mean 1450 ms -- 6.1x to 9.4x the ~100-200ms "feels instant" budget for a human clicker. No ROI is close to instant: template matching alone already exceeds that budget on every ROI. The bank matched is a single template (scales=(1.0,), n_angles=1, flips=(False,)) -- a richer augmentation bank would scale stage 3 roughly linearly, making it an even larger share of total latency, not a smaller one.
